In [0]:
displayHTML("<h1>hey, Let's start project3</h1>")

hey, Let's start project3

In [0]:
data_sample={"InvoiceNumber":"27080561","CreatedTime":1595689030008,"StoreID":"STR6347","PosID":"POS622","CashierID":"OAS793","CustomerType":"PRIME","CustomerCardNo":"5697125813","TotalAmount":9396.0,"NumberOfItems":4,"PaymentMethod":"CARD","TaxableAmount":9396.0,"CGST":234.9,"SGST":234.9,"CESS":11.745000000000001,"DeliveryType":"HOME-DELIVERY","DeliveryAddress":{"AddressLine":"295-7690 At Street","City":"Shahjahanpur","State":"Uttar Pradesh","PinCode":"228410","ContactNumber":"4624129756"},"InvoiceLineItems":[{"ItemCode":"593","ItemDescription":"Hanging curtains","ItemPrice":1896.0,"ItemQty":2,"TotalValue":3792.0},{"ItemCode":"423","ItemDescription":"Quilt","ItemPrice":1485.0,"ItemQty":1,"TotalValue":1485.0},{"ItemCode":"318","ItemDescription":"Bofinger chair","ItemPrice":1119.0,"ItemQty":1,"TotalValue":1119.0},{"ItemCode":"548","ItemDescription":"Cartel clock","ItemPrice":1500.0,"ItemQty":2,"TotalValue":3000.0}]}

In [0]:
display(data_sample)

{'InvoiceNumber': '27080561',
 'CreatedTime': 1595689030008,
 'StoreID': 'STR6347',
 'PosID': 'POS622',
 'CashierID': 'OAS793',
 'CustomerType': 'PRIME',
 'CustomerCardNo': '5697125813',
 'TotalAmount': 9396.0,
 'NumberOfItems': 4,
 'PaymentMethod': 'CARD',
 'TaxableAmount': 9396.0,
 'CGST': 234.9,
 'SGST': 234.9,
 'CESS': 11.745000000000001,
 'DeliveryType': 'HOME-DELIVERY',
 'DeliveryAddress': {'AddressLine': '295-7690 At Street',
  'City': 'Shahjahanpur',
  'State': 'Uttar Pradesh',
  'PinCode': '228410',
  'ContactNumber': '4624129756'},
 'InvoiceLineItems': [{'ItemCode': '593',
   'ItemDescription': 'Hanging curtains',
   'ItemPrice': 1896.0,
   'ItemQty': 2,
   'TotalValue': 3792.0},
  {'ItemCode': '423',
   'ItemDescription': 'Quilt',
   'ItemPrice': 1485.0,
   'ItemQty': 1,
   'TotalValue': 1485.0},
  {'ItemCode': '318',
   'ItemDescription': 'Bofinger chair',
   'ItemPrice': 1119.0,
   'ItemQty': 1,
   'TotalValue': 1119.0},
  {'ItemCode': '548',
   'ItemDescription': 'Cartel 

- process the invoice data using stream processing

In [0]:
project_dir = "dbfs:/FileStore/project3/"

In [0]:
from pyspark.sql.functions import asc, desc, lower, upper, trim, split, expr

In [0]:
class invoiceStreamProcessing:
    def __init__(self,project_dir:str):
        self.dataset_dir="dataset/"
        self.landing_zone_dir="landing/"
        self.checkpoint_dir="checkpoint/"
        self.load_table_name="proj3_invoice_table"
        self.project_dir=project_dir
        self.result_df=None
        self.streamingQuery=None
        self.processingTime="30 seconds" #seconds
    
    def cleanup_n_setups(self):
        # drop table and its data file storage directory
        spark.sql(f"drop table if exists {self.load_table_name}")
        dbutils.fs.rm("/user/hive/warehouse/"+self.load_table_name,True)

        # remove checkpoint and landing_zone and create them new

        dbutils.fs.rm(self.project_dir+self.checkpoint_dir, True)
        dbutils.fs.rm(self.project_dir+self.landing_zone_dir, True)

        dbutils.fs.mkdirs(self.project_dir+self.checkpoint_dir)
        dbutils.fs.mkdirs(self.project_dir+self.landing_zone_dir)

        print("CLEANUP & SETUP RES: completed successfully !")

    def ingest_data_to_landing_zone(self, fileName:str):
        dbutils.fs.cp(self.project_dir+self.dataset_dir+fileName, self.project_dir+self.landing_zone_dir)
        print(f"INGESTION RES: {fileName} has been successfully ingested from dataset dir to landing zone dir")
    
    def get_invoice_schema(self)->str:
        return """ InvoiceNumber string, CreatedTime bigint, StoreID string, PosID string, CashierID string, CustomerType string, CustomerCardNo string, TotalAmount double, SGST double, CESS double, DeliveryType string,
        DeliveryAddress struct< AddressLine string, City string, State string, PinCode string, ContactNumber string>,
        InvoiceLineItems array<struct<ItemCode string, ItemDescription string, ItemPrice double, ItemQty bigint, TotalValue double>> """

    def extract_data_from_landing_zone(self):
        raw_df=spark.readStream.format("json").schema(self.get_invoice_schema()).load(self.project_dir+self.landing_zone_dir+"*.json")
        print("EXTRACTION RES: data file has been successfully extracted from landing zone")
        self.result_df=raw_df
        return raw_df
    
    def transform_data(self,extracted_df):
        exploded_df=extracted_df.selectExpr("InvoiceNumber", "CreatedTime","StoreID", "PosID", "CashierID" ," CustomerType" ," CustomerCardNo" , "TotalAmount", "SGST", "CESS","DeliveryType",
        "DeliveryAddress.AddressLine", "DeliveryAddress.City", "DeliveryAddress.State", "DeliveryAddress.PinCode", "DeliveryAddress.ContactNumber",
        "explode(InvoiceLineItems) as LineItem")
        transformed_df=exploded_df.withColumn("ItemCode",expr("LineItem.ItemCode")).withColumn("ItemDescription",expr("LineItem.ItemDescription")).withColumn("ItemPrice",expr("LineItem.ItemPrice")).withColumn("ItemQty",expr("LineItem.ItemQty")).withColumn("TotalValue",expr("LineItem.TotalValue")).drop("LineItem")
        print("TRANSFORMATION RES: successfull !")
        self.result_df=transformed_df
        
    def load_data(self):
        self.streamingQuery=self.result_df.writeStream.format("delta").option("checkpointLocation",self.project_dir+self.checkpoint_dir).outputMode("append").option("maxFilesPerTrigger",1).trigger(processingTime=self.processingTime).toTable(self.load_table_name)
        print("LOAD RES: data loaded successfully\n","*"*10,"DATASTREAM STARTED", "*"*10)

    def getStreamingQuery(self):
        return self.streamingQuery

    def stopStreamingQuery(self):      
        # self.streamingQuery.stop()
        self.streamingQuery.stop()
        res=self.streamingQuery.status
        if res["message"]=="Stopped":
            print("STREAM STOP: Successfull !")
        else:
            print("STREAM STOP: Unuccessfull !")



In [0]:
# import time

In [0]:
# sleep_time=30 # in seconds
# invStrmPro=invoiceStreamProcessing(project_dir=project_dir)
# invStrmPro.cleanup_n_setups()
# invStrmPro.ingest_data_to_landing_zone("invoices_1.json")
# raw_df=invStrmPro.extract_data_from_landing_zone()
# invStrmPro.transform_data(raw_df)
# invStrmPro.load_data()

# invStrmPro.ingest_data_to_landing_zone("invoices_2.json")
# invStrmPro.ingest_data_to_landing_zone("invoices_3.json")
# time.sleep(sleep_time)
# res=invStrmPro.stopStreamingQuery()
# if res:
#     print("STREAMING STOPPED !")
# else:
#     print("SOMETHING WENT WRONG !")

### project3 completed !